# Classifier evaluation

This notebook renders report figures from exported evaluation metrics only.

In [ ]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebooks.utils.report_diagrams_utils import (available_metric_rows, normalized_confusion_matrix, report_output_paths, save_report_figure, save_report_table)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
OUTPUTS = report_output_paths(DATA_ROOT)
RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
RESULT_ROOT = DATA_ROOT / "experiment-results/classification/evaluations" / RUN_ID
ANALYSIS_ROOT = DATA_ROOT / "experiment-results/classification/analysis/complete-analysis-feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100/output"
sns.set_theme(style="whitegrid", context="paper")

In [ ]:
metrics = pd.read_csv(RESULT_ROOT / "evaluation_metrics.csv")
metrics = metrics.loc[metrics["partition"].eq("test")].copy()
metrics = available_metric_rows(metrics, "macro_f1")
save_report_table(metrics, OUTPUTS.tables / "classifier_evaluation_metrics.csv")
for name in ("binary_metrics", "robustness_metrics", "operating_points"):
    path = ANALYSIS_ROOT / f"{name}.csv"
    if path.is_file():
        save_report_table(pd.read_csv(path), OUTPUTS.tables / f"{name}.csv")

In [ ]:
summary = metrics.groupby(["scorer_llm", "feature_set", "dataset"], as_index=False)[["roc_auc", "pr_auc", "macro_f1", "balanced_accuracy"]].mean()
save_report_table(summary, OUTPUTS.tables / "classifier_metric_summary.csv")
long = summary.melt(id_vars=["scorer_llm", "feature_set", "dataset"], var_name="metric", value_name="score")
figure, axis = plt.subplots(figsize=(12, 6), constrained_layout=True)
sns.barplot(data=long, x="dataset", y="score", hue="metric", errorbar=None, ax=axis)
axis.set(title="Classifier performance on held-out test rows", ylim=(0, 1))
save_report_figure(figure, OUTPUTS.figures / "classifier_metric_comparison")
plt.close(figure)

In [ ]:
confusion = normalized_confusion_matrix(metrics)
save_report_table(confusion.rename_axis("true_class").reset_index(), OUTPUTS.tables / "classifier_confusion_matrix_normalized.csv")
figure, axis = plt.subplots(figsize=(5, 4), constrained_layout=True)
sns.heatmap(confusion, annot=True, fmt=".3f", vmin=0, vmax=1, cmap="Blues", ax=axis)
axis.set(title="Normalized classifier confusion matrix", xlabel="Predicted class", ylabel="True class")
save_report_figure(figure, OUTPUTS.figures / "classifier_confusion_matrix_normalized")
plt.close(figure)